# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an example for loading and exploring a FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

**https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json**

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load schema metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and schema
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"Dataset name: {meta.name}")
print(f"Description: {meta.description}")

## 2. Data Overview
Review available record sets and fields by their `@id` to understand the structure of this dataset schema.

In [ ]:
# Get the available RecordSet `@id`s
record_sets = meta.record_sets
if not record_sets:
    # Try loading from the full metadata dictionary, if .record_sets is empty
    import requests
    croissant_dict = requests.get(croissant_url).json()
    record_sets = [r['@id'] for r in croissant_dict.get('recordSet', [])]

if record_sets:
    print("The dataset contains the following RecordSets (by @id):")
    for rs_id in record_sets:
        print(f"- {rs_id}")
    # Examine the first RecordSet as an example
    example_record_set_id = record_sets[0]
else:
    # If the record_sets is still empty, print a message
    print("No RecordSets available in this schema.")
    example_record_set_id = None

# Show fields for each record set by @id
if example_record_set_id:
    print(f"\nFields in RecordSet {example_record_set_id}:")
    # Try to get fields using Croissant API, else fallback to JSON
    try:
        recordset_obj = dataset.get_record_set(example_record_set_id)
        field_ids = [f["@id"] for f in recordset_obj.fields]
    except Exception:
        # Manual extraction from JSON
        for r in croissant_dict['recordSet']:
            if r['@id'] == example_record_set_id:
                field_ids = [f["@id"] for f in r.get('field',[])]
                break
        else:
            field_ids = []
    for field_id in field_ids:
        print(f"  - {field_id}")
else:
    field_ids = []

## 3. Data Extraction
Load records from a specific record set as a pandas DataFrame, referencing all record set and field entities by their `@id`.

In [ ]:
# Load all record sets into pandas DataFrames using their @id
dataframes = {}
for record_set_id in record_sets:
    try:
        # The .records API yields Python dicts for each record
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from RecordSet {record_set_id}.")
        else:
            print(f"No records found in {record_set_id}.")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

if dataframes:
    # Pick the first available dataframe for demonstration
    demo_record_set_id = list(dataframes.keys())[0]
    print(f"\nAvailable columns/fields in RecordSet {demo_record_set_id}:")
    print(dataframes[demo_record_set_id].columns.tolist())
    display(dataframes[demo_record_set_id].head())
else:
    print("No tabular data could be loaded from any record set.")
    demo_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
We will perform simple filtering, normalization, and grouping operations using field `@id` references.
Let's assume the dataset contains an age-related field with `@id` 'age' and a grouping field, e.g., 'sex'.
Please adjust the field IDs below to match those found above.

In [ ]:
# Example EDA: Filter records, normalize numeric fields, group by categorical

# Adjust these as needed to match your dataset's real field @ids:
numeric_field_id = None
group_field_id = None

if demo_record_set_id is not None:
    demo_df = dataframes[demo_record_set_id]
    # Attempt to infer a numeric field (try common field names)
    candidate_numeric_fields = [c for c in demo_df.columns if any(k in c.lower() for k in ['age', 'interval', 'duration', 'number', 'count']) or pd.api.types.is_numeric_dtype(demo_df[c])]
    if candidate_numeric_fields:
        numeric_field_id = candidate_numeric_fields[0]
    # Attempt to infer a group field (e.g. 'sex', 'gender')
    candidate_group_fields = [c for c in demo_df.columns if any(k in c.lower() for k in ['sex', 'gender', 'location', 'site'])]
    if candidate_group_fields:
        group_field_id = candidate_group_fields[0]

if numeric_field_id:
    print(f"Numeric field selected: {numeric_field_id}")
    threshold = 50  # Change this threshold as appropriate
    filtered_df = demo_df[pd.to_numeric(demo_df[numeric_field_id], errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Grouping
    if group_field_id and group_field_id in demo_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped (mean) data by {group_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric field found for demonstration. Please inspect the available fields and adjust the script.")

## 5. Visualization
Visualize the distribution of a key numeric field or the relationship between two fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field, if available
if demo_record_set_id is not None and numeric_field_id and numeric_field_id in demo_df:
    plt.figure(figsize=(8,5))
    sns.histplot(pd.to_numeric(demo_df[numeric_field_id], errors='coerce').dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    # Violin plot by group
    if group_field_id and group_field_id in demo_df:
        plt.figure(figsize=(10,5))
        sns.violinplot(x=demo_df[group_field_id], y=pd.to_numeric(demo_df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No suitable numeric or grouping fields found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a FAIR² tabular biomedical dataset using the `mlcroissant` library. The exploration covered:
- How to reference record sets and fields using their `@id` as per the Croissant specification
- How to inspect schema, load data, and perform basic filtering, normalization, grouping, and visualization operations

Please modify the names of fields to match those in your dataset for custom analysis. For further analysis, consult the [FAIR² standard](https://mlcommons.github.io/croissant/) and the [mlcroissant](https://pypi.org/project/mlcroissant/) documentation.